In [1]:
from pathlib import Path
import pandas as pd

# Set the source and output folders for the 2026 voter-registration snapshot.
# We will stack the municipality files into one current ward-level registration table.

source_dir = Path(
    r"C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\interim\Voter Registration_KZN_cleaned"
)

output_dir = Path(
    r"C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\processed\05_voter_registration_2026"
)

output_dir.mkdir(parents=True, exist_ok=True)

print("Source folder exists:", source_dir.exists())
print("Output folder:", output_dir)

Source folder exists: True
Output folder: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\processed\05_voter_registration_2026


Now we verify exactly what is in the folder before stacking the files. We want to confirm:

the number of CSV files,
that the expected 41 files are present,
that they have the same structure,
and that there aren't unexpected files being included.


In [2]:
# Inventory the cleaned registration files before combining them.
# This confirms we have the expected municipality-level files and no unexpected inputs.

registration_files = sorted(source_dir.glob("*.csv"))

print("CSV files found:", len(registration_files))
print("\nFiles:")

for file in registration_files:
    print(file.name)

CSV files found: 44

Files:
iec_kzn_ethekwini_ward_registration_2026_clean.csv
iec_kzn_kzn212___umdoni_ward_registration_2026_clean.csv
iec_kzn_kzn213___umzumbe_ward_registration_2026_clean.csv
iec_kzn_kzn214___umuziwabantu_ward_registration_2026_clean.csv
iec_kzn_kzn216___ray_nkonyeni_ward_registration_2026_clean.csv
iec_kzn_kzn221___umshwathi_ward_registration_2026_clean.csv
iec_kzn_kzn222___umngeni_ward_registration_2026_clean.csv
iec_kzn_kzn223___mpofana_ward_registration_2026_clean.csv
iec_kzn_kzn224___impendle_ward_registration_2026_clean.csv
iec_kzn_kzn225___msunduzi_ward_registration_2026_clean.csv
iec_kzn_kzn226___mkhambathini_ward_registration_2026_clean.csv
iec_kzn_kzn227___richmond_ward_registration_2026_clean.csv
iec_kzn_kzn235___okhahlamba_ward_registration_2026_clean.csv
iec_kzn_kzn237___inkosi_langalibalele_ward_registration_2026_clean.csv
iec_kzn_kzn238___alfred_duma_ward_registration_2026_clean.csv
iec_kzn_kzn241___endumeni_ward_registration_2026_clean.csv
iec_kzn_kzn

In [3]:
# Check that all 44 municipality files use the same structure before stacking them.
# A consistent schema lets us combine the current 2026 registration snapshot safely.

file_schemas = {}

for file in registration_files:
    df_check = pd.read_csv(file)
    file_schemas[file.name] = tuple(df_check.columns)

unique_schemas = set(file_schemas.values())

print("Files checked:", len(file_schemas))
print("Unique column structures:", len(unique_schemas))

if len(unique_schemas) == 1:
    print("Schema validation: PASSED")
    print("Columns:")
    print(list(next(iter(unique_schemas))))
else:
    print("Schema validation: FAILED")
    print("\nDifferent schemas found:")
    for filename, schema in file_schemas.items():
        print(f"\n{filename}")
        print(schema)

Files checked: 44
Unique column structures: 1
Schema validation: PASSED
Columns:
['ward', 'voting_districts', 'registered_voters']


In [4]:
# Inspect sample records and data types before combining the 44 files.
# This confirms the ward identifiers and voter counts are usable for the 2026 snapshot.

sample_file = registration_files[0]
sample_df = pd.read_csv(sample_file)

print("Sample file:", sample_file.name)
print("\nShape:", sample_df.shape)
print("\nData types:")
print(sample_df.dtypes)

print("\nFirst 10 rows:")
display(sample_df.head(10))

print("\nMissing values:")
display(sample_df.isna().sum())

Sample file: iec_kzn_ethekwini_ward_registration_2026_clean.csv

Shape: (112, 3)

Data types:
ward                 int64
voting_districts     int64
registered_voters    int64
dtype: object

First 10 rows:


,ward,voting_districts,registered_voters
0,59500001,14,19964
1,59500002,21,21172
2,59500003,13,17103
3,59500004,9,20987
4,59500005,5,14950
5,59500006,7,18454
6,59500007,7,16867
7,59500008,12,20847
8,59500009,10,20985
9,59500010,8,20505



Missing values:


ward                 0
voting_districts     0
registered_voters    0
dtype: int64

In [5]:
# Add municipality and source-file identifiers before stacking the registration data.
# This preserves the geographic origin of every 2026 ward record for validation and auditing.

registration_parts = []

for file in registration_files:
    df = pd.read_csv(file)

    municipality = file.stem.replace(
        "iec_kzn_", ""
    ).replace(
        "_ward_registration_2026_clean", ""
    )

    df["municipality"] = municipality
    df["source_file"] = file.name

    registration_parts.append(df)

print("Files prepared:", len(registration_parts))
print("Columns after adding identifiers:")
print(registration_parts[0].columns.tolist())

Files prepared: 44
Columns after adding identifiers:
['ward', 'voting_districts', 'registered_voters', 'municipality', 'source_file']


In [6]:
# Check the municipality names extracted from the filenames before stacking.
# This makes sure every registration record keeps the correct municipality identity.

municipality_values = sorted(
    df["municipality"].iloc[0] for df in registration_parts
)

print("Municipality values extracted:", len(municipality_values))
print("\nExtracted municipality identifiers:")

for municipality in municipality_values:
    print(municipality)

Municipality values extracted: 44

Extracted municipality identifiers:
ethekwini
kzn212___umdoni
kzn213___umzumbe
kzn214___umuziwabantu
kzn216___ray_nkonyeni
kzn221___umshwathi
kzn222___umngeni
kzn223___mpofana
kzn224___impendle
kzn225___msunduzi
kzn226___mkhambathini
kzn227___richmond
kzn235___okhahlamba
kzn237___inkosi_langalibalele
kzn238___alfred_duma
kzn241___endumeni
kzn242___nqutu
kzn244___umsinga
kzn245___umvoti
kzn252___newcastle
kzn253___emadlangeni
kzn254___dannhauser
kzn261___edumbe
kzn262___uphongolo
kzn263___abaqulusi
kzn265___nongoma
kzn266___ulundi
kzn271___umhlabuyalingana
kzn272___jozini
kzn275___inkosi_umtubatuba
kzn276___big_five_hlabisa
kzn281___umfolozi
kzn282___umhlathuze
kzn284___umlalazi
kzn285___mthonjaneni
kzn286___nkandla
kzn291___mandeni
kzn292___kwadukuza
kzn293___ndwedwe
kzn294___maphumulo
kzn433___greater_kokstad
kzn434___johannes_phumani_pungula
kzn435___umzimkhulu
kzn436___dr._nkosazana_dlamini_zuma


In [7]:
# Convert the filename-based municipality identifiers into readable municipality names.
# The original source_file is kept so every record can still be traced back to its input.

for df in registration_parts:
    df["municipality"] = (
        df["municipality"]
        .str.replace(r"^kzn\d+___", "", regex=True)
        .str.replace("_", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.title()
    )

    df.loc[df["municipality"].eq("Ethekwini"), "municipality"] = "eThekwini"

municipality_names = sorted(
    df["municipality"].iloc[0] for df in registration_parts
)

print("Readable municipality names:", len(municipality_names))
print("\nMunicipalities:")

for municipality in municipality_names:
    print(municipality)

Readable municipality names: 44

Municipalities:
Abaqulusi
Alfred Duma
Big Five Hlabisa
Dannhauser
Dr. Nkosazana Dlamini Zuma
Edumbe
Emadlangeni
Endumeni
Greater Kokstad
Impendle
Inkosi Langalibalele
Inkosi Umtubatuba
Johannes Phumani Pungula
Jozini
Kwadukuza
Mandeni
Maphumulo
Mkhambathini
Mpofana
Msunduzi
Mthonjaneni
Ndwedwe
Newcastle
Nkandla
Nongoma
Nqutu
Okhahlamba
Ray Nkonyeni
Richmond
Ulundi
Umdoni
Umfolozi
Umhlabuyalingana
Umhlathuze
Umlalazi
Umngeni
Umshwathi
Umsinga
Umuziwabantu
Umvoti
Umzimkhulu
Umzumbe
Uphongolo
eThekwini


In [8]:
# Stack all 44 municipality files into one current 2026 registration table.
# Each row keeps its ward, voter count, municipality, and original source file.

voter_registration_2026 = pd.concat(
    registration_parts,
    ignore_index=True
)

print("Files combined:", len(registration_parts))
print("Rows:", len(voter_registration_2026))
print("Columns:", len(voter_registration_2026.columns))

display(voter_registration_2026.head())

Files combined: 44
Rows: 921
Columns: 5


,ward,voting_districts,registered_voters,municipality,source_file
0,59500001,14,19964,eThekwini,iec_kzn_ethekwini_ward_registration_2026_clean...
1,59500002,21,21172,eThekwini,iec_kzn_ethekwini_ward_registration_2026_clean...
2,59500003,13,17103,eThekwini,iec_kzn_ethekwini_ward_registration_2026_clean...
3,59500004,9,20987,eThekwini,iec_kzn_ethekwini_ward_registration_2026_clean...
4,59500005,5,14950,eThekwini,iec_kzn_ethekwini_ward_registration_2026_clean...


In [9]:
# Validate the combined 2026 registration table before saving it.
# We check for duplicate municipality-ward records and invalid voter-registration values.

duplicate_wards = voter_registration_2026.duplicated(
    subset=["municipality", "ward"]
).sum()

missing_values = voter_registration_2026.isna().sum().sum()

invalid_voters = (
    voter_registration_2026["registered_voters"] < 0
).sum()

print("Rows:", len(voter_registration_2026))
print("Duplicate municipality-ward records:", duplicate_wards)
print("Total missing values:", missing_values)
print("Negative registered-voter values:", invalid_voters)

if (
    duplicate_wards == 0
    and missing_values == 0
    and invalid_voters == 0
):
    print("\nRegistration integrity validation: PASSED")
else:
    print("\nRegistration integrity validation: FAILED")

Rows: 921
Duplicate municipality-ward records: 0
Total missing values: 0
Negative registered-voter values: 0

Registration integrity validation: PASSED


In [10]:
# Check municipality coverage and ward counts before saving the 2026 snapshot.
# This confirms all 44 municipalities contributed records to the final table.

municipality_coverage = (
    voter_registration_2026
    .groupby("municipality")
    .agg(
        wards=("ward", "nunique"),
        registered_voters=("registered_voters", "sum")
    )
    .sort_values("municipality")
)

print("Municipalities represented:", municipality_coverage.shape[0])
print("Total wards:", municipality_coverage["wards"].sum())
print("Total registered voters:", municipality_coverage["registered_voters"].sum())

display(municipality_coverage)


Municipalities represented: 44
Total wards: 921
Total registered voters: 6030969


,wards,registered_voters
municipality,,
Abaqulusi,24,112610
Alfred Duma,37,181335
Big Five Hlabisa,14,63355
Dannhauser,13,57185
Dr. Nkosazana Dlamini Zuma,15,61948
Edumbe,10,44608
Emadlangeni,6,17694
Endumeni,8,35924
Greater Kokstad,11,48778


In [11]:
# Save the validated 2026 voter-registration snapshot to the processed layer.
# This becomes the current registration input that can later be joined to the ward master dataset.

output_path = output_dir / "voter_registration_2026.csv"

voter_registration_2026.to_csv(
    output_path,
    index=False
)

print("Voter registration panel saved.")
print("Output file:", output_path)
print("Rows saved:", len(voter_registration_2026))
print("Columns saved:", len(voter_registration_2026.columns))

Voter registration panel saved.
Output file: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\processed\05_voter_registration_2026\voter_registration_2026.csv
Rows saved: 921
Columns saved: 5


In [12]:
# Re-open the saved file and confirm the processed snapshot was written correctly.
# This final check makes sure the saved file still contains all 921 validated ward records.

saved_registration = pd.read_csv(output_path)

print("File exists:", output_path.exists())
print("Rows loaded:", len(saved_registration))
print("Columns loaded:", len(saved_registration.columns))
print("Columns:", saved_registration.columns.tolist())

if (
    output_path.exists()
    and len(saved_registration) == 921
    and len(saved_registration.columns) == 5
):
    print("\nVoter registration save validation: PASSED")
else:
    print("\nVoter registration save validation: FAILED")

File exists: True
Rows loaded: 921
Columns loaded: 5
Columns: ['ward', 'voting_districts', 'registered_voters', 'municipality', 'source_file']

Voter registration save validation: PASSED
